# Notebook 09 — Clean replay behavioral evaluation (Gates 6 & 7)

Test the frozen text-only context in a FRESH subprocess with no intervention. This notebook shells out to scripts/clean_replay_eval.py (import-guarded).

In [1]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


FAST_DEV_RUN= False RUN_EXPENSIVE= True pkg 0.1.0


## 1. Configuration and run manifest

In [2]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="09", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


run_id: 20260717_220715_smoke_eagle_09_nogit_0


## 2. Preflight assertions

In [3]:
print('preflight: package + numpy import OK')

preflight: package + numpy import OK


## 3. Load or compute cached artifacts
Invoke the clean-replay self-test via subprocess; confirm the import guard holds.

In [4]:
import subprocess, sys
r = subprocess.run([sys.executable, str(REPO / "scripts/clean_replay_eval.py"), "--self-test"],
                   capture_output=True, text=True)
print(r.stdout.strip()); print(r.stderr.strip()[-300:] if r.stderr else "")
clean_ok = r.returncode == 0 and "PASS" in r.stdout

clean_replay self-test PASS (import guard clean, scoring correct, F_T=1.719)



## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables
See printed tables above; plotting helpers in `subliminal_icl.plotting`.

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [5]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("clean_process_selftest", clean_ok, "import guard + scoring OK in fresh process")]
gs = run_gate_checks("gate_07_clean_replay", "Clean text-only replay", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


,check,result,detail
0,clean_process_selftest,PASS,import guard + scoring OK in fresh process


GATE gate_07_clean_replay -> PASS


## 7. Write checkpoint report
Written by the gate cell when not in FAST_DEV_RUN.